# The Beatles, Part D: Charts and Graphs

**Beatles notebooks:** [A. Clean Data](A_Beatles_Clean_Data.ipynb) · [B. Tidy Data](B_Beatles_Tidy_Data.ipynb) · [C. Find, Filter, Group](C_Beatles_Find_Filter_Group.ipynb) · **D. Charts and Graphs**

**The big picture.** A chart turns a table into a pattern the eye can grasp at once: a trend over time, a cluster, an outlier. But every chart is also an **argument**. What we count, how we group, what we leave out, and even the colors we choose all shape what a reader sees. Here we tour the main chart types in [Plotly Express](https://plotly.com/python/plotly-express/) and ask which kind of question each one is good for:

| Your question is about... | Try a... |
|---|---|
| comparing amounts across categories | bar chart |
| change over time | line chart |
| how values are spread out | histogram or box plot |
| the relationship between two measurements | scatter or bubble plot |
| a grid of values (e.g. correlations) | heatmap |
| a profile across several features | radar chart |

**In this notebook you will learn to:**
* prepare data for a chart with `groupby()` and `melt()`
* build each chart type with one basic pattern: `fig = px.chart(...)`, then `fig.show()`
* customize titles, labels, colors, sizes, and hover information
* read a chart critically, and tell a pattern from an accident of the data

**Input:** `beatles_tidy_genre.pkl` (from Notebook B). Plotly charts are **interactive**: hover for details, drag to zoom, double-click to reset, and click legend items to hide or show them.

**Keep in mind:** each chart here is a starting point, not a conclusion. After each one you'll find **questions to explore**. Interpreting what you see is up to you.

**Contents of this Notebook**

- **1.** Bar Charts
- **2.** Line Charts
- **3.** Histograms and Box Plots
- **4.** Scatter and Bubble Plots
- **5.** Heatmaps
- **6.** Radar (Polar) Charts
- **7.** Customizing Your Charts: Quick Reference

In [ ]:
import pandas as pd
import plotly.express as px

beatles_tidy = pd.read_pickle('beatles_tidy_genre.pkl')

# one row per song, for questions about songs rather than genres (see Notebook B)
songs = beatles_tidy.drop_duplicates(subset='title')

print('tidy rows:', len(beatles_tidy), '   songs:', len(songs))

### The basic pattern

Almost every Plotly Express chart follows the same three steps:

```python
fig = px.chart_type(dataframe, x='column', y='column', color='column', title='...')   # 1. define
fig.update_layout(width=800, height=500)                                              # 2. adjust (optional)
fig.show()                                                                            # 3. show
```

A few settings we'll reuse: the three main songwriters, the studio albums in release order, and a fixed color for each songwriter, so that Lennon is always the same color in every chart.

In [ ]:
big_three = ['Lennon', 'McCartney', 'Harrison']
songwriter_colors = {'Lennon': '#2a78d6', 'McCartney': '#eb6834', 'Harrison': '#1baf7a'}

studio_albums = ['Please Please Me', 'With the Beatles', "A Hard Day's Night", 'Beatles for Sale',
                 'Help!', 'Rubber Soul', 'Revolver', "Sgt. Pepper's Lonely Hearts Club Band",
                 'Magical Mystery Tour', 'The Beatles', 'Yellow Submarine', 'Abbey Road', 'Let It Be']

features = ['danceability', 'energy', 'valence', 'acousticness', 'speechiness', 'liveness']

## 1. Bar Charts

Bar charts compare a **number across categories**. [Plotly Documentation](https://plotly.com/python/bar-charts/)

### Simple bar chart: how many songs per genre?

We count **distinct songs** per genre in the tidy data (leaving out the umbrella label `Pop/Rock`). A horizontal bar (`orientation='h'`) keeps long genre names readable.

In [ ]:
genre_counts = (beatles_tidy[beatles_tidy['genre'] != 'Pop/Rock']
                .groupby('genre')['title'].nunique()
                .reset_index(name='songs')
                .nlargest(15, 'songs'))

fig = px.bar(genre_counts, x='songs', y='genre', orientation='h',
             title='Most Common Genres in Beatles Songs (excluding Pop/Rock)',
             labels={'songs': 'Number of songs', 'genre': ''})
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, width=750, height=500)
fig.show()

### Stacked bar chart: genres over time

Adding `color=` splits each bar into segments. Here: the six most common genres, year by year.

In [ ]:
top_genres = genre_counts['genre'].head(6).tolist()

genre_year = (beatles_tidy[beatles_tidy['genre'].isin(top_genres)]
              .groupby(['year', 'genre'])['title'].nunique()
              .reset_index(name='songs'))

fig = px.bar(genre_year, x='year', y='songs', color='genre',
             title='Top Six Genres by Year',
             labels={'songs': 'Number of songs', 'year': 'Year', 'genre': 'Genre'})
fig.update_layout(width=850, height=500)
fig.show()

### Grouped bar chart: audio features by album

Passing a **list** of columns to `y=` makes one bar per feature; `barmode='group'` places them side by side instead of stacking them. `category_orders` keeps the albums in release order.

In [ ]:
album_features = (songs[songs['album'].isin(studio_albums)]
                  .groupby('album')[['energy', 'valence', 'acousticness']].mean()
                  .reset_index())

fig = px.bar(album_features, x='album', y=['energy', 'valence', 'acousticness'],
             barmode='group',
             category_orders={'album': studio_albums},
             title='Average Audio Features by Studio Album',
             labels={'value': 'Average score (0–1)', 'variable': 'Feature', 'album': ''})
fig.update_xaxes(tickangle=45)
fig.update_layout(width=950, height=550)
fig.show()

**Questions to explore**

* Which genres dominate, and which are rare? Are the rare ones concentrated in certain years?
* Does the genre mix change over time? Where would you draw the boundaries between "periods"?
* Which albums stand out from their neighbors? Try other features (`danceability`, `speechiness`, `liveness`).
* Try `color='songwriter_group'` on the genre bar chart. What changes?

## 2. Line Charts

Line charts show **change over an ordered axis**, usually time. [Docs](https://plotly.com/python/line-charts/)

Plotly Express likes **long** data for multiple lines, so we `melt` the yearly averages (see Notebook B): one row per year × feature.

In [ ]:
yearly = songs[songs['year'] <= 1970].groupby('year')[['energy', 'valence', 'acousticness']].mean().reset_index()

yearly_long = yearly.melt(id_vars='year', var_name='feature', value_name='average')

fig = px.line(yearly_long, x='year', y='average', color='feature', markers=True,
              title='Average Audio Features by Year, 1958–1970',
              labels={'average': 'Average score (0–1)', 'year': 'Year', 'feature': 'Feature'})
fig.update_layout(width=850, height=500)
fig.show()

**Questions to explore**

* Which features change the most over the Beatles' career? Which stay steady?
* Some years have very few songs. How might that affect the averages? (Hint: add a `count` to the `groupby` and check.)
* Make the same chart with one line per **songwriter** instead of per feature (`color='songwriter_group'`).

## 3. Histograms and Box Plots

These show the **distribution** of a single numeric variable: where values cluster, how spread out they are, and which songs are outliers.

### Histogram
[Docs](https://plotly.com/python/histograms/)

In [ ]:
fig = px.histogram(songs, x='energy', nbins=20, color='mode_name', barmode='overlay',
                   title='Distribution of Energy: Major vs. Minor Keys',
                   labels={'energy': 'Energy (0 = calm, 1 = intense)', 'mode_name': 'Mode'})
fig.update_layout(width=800, height=450)
fig.show()

### Box plot
A box shows the median (middle line), the middle 50% of songs (the box), and the range (whiskers). `points='all'` also draws each song; hover to see which one. [Docs](https://plotly.com/python/box-plots/)

In [ ]:
fig = px.box(songs[songs['songwriter_group'].isin(big_three)],
             x='songwriter_group', y='valence', color='songwriter_group',
             color_discrete_map=songwriter_colors, points='all',
             hover_data=['title', 'year'],
             title='Valence (Musical "Positiveness") by Songwriter',
             labels={'songwriter_group': 'Songwriter', 'valence': 'Valence (0–1)'})
fig.update_layout(width=700, height=500, showlegend=False)
fig.show()

**Questions to explore**

* Is there a difference between major- and minor-key songs? Is it as big as you expected?
* How do the three songwriters compare in the middle and at the extremes? Hover over the outliers: do they surprise you?
* Swap in other features and other groupings: `x='is_cover'`, `x='billboard_top50'`, or the `period` bins you made with `pd.cut()` in Notebook C.

## 4. Scatter and Bubble Plots

Scatter plots show the **relationship between two numeric variables**, one dot per song. [Docs](https://plotly.com/python/line-and-scatter/)

### Scatter plot with hover data

In [ ]:
fig = px.scatter(songs[songs['songwriter_group'].isin(big_three)],
                 x='danceability', y='energy', color='songwriter_group',
                 color_discrete_map=songwriter_colors,
                 hover_data=['title', 'album', 'year'],
                 title='Danceability vs. Energy',
                 labels={'danceability': 'Danceability (0–1)', 'energy': 'Energy (0–1)',
                         'songwriter_group': 'Songwriter'})
fig.update_layout(width=800, height=600)
fig.show()

> **Trend lines:** add `trendline='ols'` to `px.scatter()` to draw a line of best fit. This requires the `statsmodels` library (`pip install statsmodels`).

### Then vs. now: chart rank and Spotify popularity

Only the 49 songs on Billboard's Top 50 list have a rank. Rank 1 is the best, so we **reverse** the x-axis to put the top hits on the right.

In [ ]:
ranked = songs[songs['billboard_top50']]

ranked_pre_1971 = ranked[ranked['year'] < 1971]

fig = px.scatter(ranked_pre_1971, x='top_50_billboard', y='popularity',
                 hover_data=['title', 'year'], color='year',
                 color_continuous_scale='Viridis',
                 title="Billboard's Top 50 Beatles Songs vs. Spotify Popularity Today",
                 labels={'top_50_billboard': 'Billboard Top 50 rank', 'popularity': 'Spotify popularity',
                         'year': 'Year'})
fig.update_traces(marker=dict(size=11, line=dict(width=1, color='DarkSlateGrey')))
fig.update_xaxes(autorange='reversed')
fig.update_layout(width=800, height=550, plot_bgcolor='white')
fig.update_xaxes(showgrid=True, gridcolor='lightgrey')
fig.update_yaxes(showgrid=True, gridcolor='lightgrey')
fig.show()

### Bubble plot: counts on a categorical grid

Setting `size=` to a count turns a scatter plot into a **bubble plot**. Here: how many songs each songwriter contributed to each genre, year by year.

In [ ]:
beatles_tidy_pre_1971 = beatles_tidy[beatles_tidy['year'] < 1971]

genre_writer_year = (beatles_tidy_pre_1971[beatles_tidy_pre_1971['genre'].isin(top_genres) &
                                  beatles_tidy_pre_1971['songwriter_group'].isin(big_three)]
                     .groupby(['year', 'genre', 'songwriter_group'])['title'].nunique()
                     .reset_index(name='songs'))

fig = px.scatter(genre_writer_year, x='year', y='genre', size='songs', color='songwriter_group',
                 color_discrete_map=songwriter_colors,
                 title='Genres by Year and Songwriter',
                 labels={'year': 'Year', 'genre': '', 'songs': 'Songs', 'songwriter_group': 'Songwriter'})
fig.update_layout(width=900, height=500)
fig.show()

**Questions to explore**

* Do the songwriters occupy different regions of the danceability/energy space? Which songs sit far from the others?
* Is a song's historical chart success a good predictor of its popularity today? Hover over the songs that break the pattern. Why might they?
* Remember: **correlation is not causation**. What *other* factors might explain Spotify popularity (film soundtracks, remasters, playlists, which version Spotify counts)?
* In the bubble plot, who moves into which genres, and when?

## 5. Heatmaps

Heatmaps use color intensity to show values in a **grid**. [Docs](https://plotly.com/python/heatmaps/)

### Correlation heatmap

`.corr()` computes the **correlation** between every pair of numeric columns: +1 means they rise together, −1 means one rises as the other falls, 0 means no linear relationship. A *diverging* color scale (two colors with white at 0) shows the direction.

In [ ]:
correlations = songs[features + ['popularity', 'tempo', 'duration']].corr()

fig = px.imshow(correlations, text_auto='.2f', color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                title='Correlations Among Audio Features')
fig.update_layout(width=700, height=650)
fig.show()

### Count heatmap: genre by songwriter

The same function can display any table, like a `pd.crosstab()` (Notebook C). For counts we use a *sequential* scale (one color, light to dark).

In [ ]:
big_three_tidy = beatles_tidy[beatles_tidy['songwriter_group'].isin(big_three) &
                              beatles_tidy['genre'].isin(genre_counts['genre'])]

genre_by_writer = pd.crosstab(big_three_tidy['genre'], big_three_tidy['songwriter_group'])

fig = px.imshow(genre_by_writer, text_auto=True, color_continuous_scale='Blues', aspect='auto',
                title='Songs per Genre and Songwriter',
                labels={'x': 'Songwriter', 'y': 'Genre', 'color': 'Songs'})
fig.update_layout(width=600, height=600)
fig.show()

**Questions to explore**

* Which pairs of features are strongly related? Are any of these relationships built into how Spotify defines its features?
* Which songwriter favors which genres? Remember that the songwriters wrote different numbers of songs: how could you make a fairer comparison? (Hint: `pd.crosstab(..., normalize='columns')`.)

## 6. Radar (Polar) Charts

Radar charts place several features around a circle, giving each song or group a "shape." [Docs](https://plotly.com/python/radar-chart/)

`px.line_polar()` needs **long** data: one row per group × feature, which is exactly what `melt` produces.

In [ ]:
writer_profile = (songs[songs['songwriter_group'].isin(big_three)]
                  .groupby('songwriter_group')[features].mean()
                  .reset_index()
                  .melt(id_vars='songwriter_group', var_name='feature', value_name='average'))

fig = px.line_polar(writer_profile, r='average', theta='feature', color='songwriter_group',
                    color_discrete_map=songwriter_colors, line_close=True,
                    title='Average Audio Profile by Songwriter',
                    labels={'songwriter_group': 'Songwriter'})
fig.update_layout(width=650, height=550)
fig.show()

The same approach works for individual songs. Change the album to explore:

In [ ]:
selected_album = 'Rubber Soul'

album_long = songs[songs['album'] == selected_album].melt(
    id_vars='title', value_vars=features, var_name='feature', value_name='value')

fig = px.line_polar(album_long, r='value', theta='feature', color='title', line_close=True,
                    title=f'Audio Features of Songs on {selected_album}')
fig.update_layout(width=800, height=650)
fig.show()

**Questions to explore**

* Averages hide variety. Does the songwriter "shape" hide big differences between individual songs?
* With many songs, a radar chart gets crowded. Click legend entries to hide songs, or pick just a few titles with `.isin([...])`.
* Compare an early album with a late one. Which features change the shape most?

## 7. Customizing Your Charts: Quick Reference

| To change... | Use |
|---|---|
| Size | `fig.update_layout(width=800, height=600)` |
| Title | `title='...'` in `px.*()`, or `fig.update_layout(title='...')` |
| Axis and legend labels | `labels={'column': 'Readable name'}` |
| Axis titles | `fig.update_layout(xaxis_title='...', yaxis_title='...')` |
| Order of categories | `category_orders={'column': [list in order]}` |
| Fixed colors per category | `color_discrete_map={'Lennon': '#2a78d6', ...}` |
| A color scheme | `color_discrete_sequence=px.colors.qualitative.Safe` |
| Continuous colors | `color_continuous_scale='Blues'` (magnitude) or `'RdBu_r'` (±) |
| Tilted tick labels | `fig.update_xaxes(tickangle=45)` |
| Reverse an axis | `fig.update_xaxes(autorange='reversed')` |
| Logarithmic axis | `log_y=True` in `px.*()` |
| Extra hover info | `hover_data=['title', 'album']` |
| Small multiples | `facet_col='column'` (one panel per category) |
| Save as interactive HTML | `fig.write_html('my_chart.html')` |


-
> **Tip on color:** use a *categorical* palette (distinct hues) for categories like songwriter or genre, a *sequential* scale (one hue, light → dark) for amounts, and a *diverging* scale (two hues, neutral middle) for values above and below zero, like correlations. Avoid rainbow scales.

### Your turn

Pick one of our research questions (or your own), and build a chart that helps answer it:

* How did the Beatles' sound change over time?
* Do Lennon, McCartney, and Harrison songs differ in genre, mood, or audio features?
* Does chart success then relate to Spotify popularity now?
* Which genres, styles, themes, and moods are most typical, and for whom? (Use `.explode('moods')`, as in Notebooks B and C, for styles, themes, or moods.)
* How do covers differ from originals?

Then write two or three sentences: what does the chart show, what *doesn't* it show, and what would you want to check next?

**Beatles notebooks:** [A. Clean Data](A_Beatles_Clean_Data.ipynb) · [B. Tidy Data](B_Beatles_Tidy_Data.ipynb) · [C. Find, Filter, Group](C_Beatles_Find_Filter_Group.ipynb) · **D. Charts and Graphs**